In [1]:
import os
from sedona.spark import SedonaContext
import geopandas as gpd
from time import time
from sedona.spark import dataframe_to_arrow
import geopandas as gpd
from sedona.spark.geoarrow import create_spatial_dataframe
from sedona.spark.maps.SedonaKepler import SedonaKepler

In [2]:
import pyspark.sql.functions as f
import matplotlib.pyplot as plt
import geopandas as gpd

In [3]:
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

In [4]:
def create_sedona_session(config_params):
    config = SedonaContext.builder()

    for key, value in config_params.items():
        config = config.config(key, value)


    sedona = SedonaContext.create(config.getOrCreate())
    sedona.sparkContext.setLogLevel("ERROR")

    return sedona
    
    
class SedonaBenchmark():
    
    def __init__(self, config: dict):
        self.sedona = create_sedona_session(config)
        
    def __enter__(self):
        return self.sedona
        
    def __exit__(self, type, value, traceback):
        self.sedona.stop()

In [5]:
from dataclasses import dataclass
import time

@dataclass
class GeometryGenerationParams:
    size: int
    geometry_type: str

    def get_scale(self, other):
        return max(self.size, other.size)

    
def create_geometries(param: GeometryGenerationParams, other_param):
    partitions = int(param.size/150_000) + 1
    scale = param.get_scale(other_param)

    df = sedona.read.format("spider").load(
        n=param.size,
        geometryType=param.geometry_type,
        maxSize=0.01,
        seed=43,
        scaleX=scale,
        scaleY=scale,
        numPartitions=partitions,
    )

    return df


def time_it_with_container(l, fn):
    start = time.time()
    fn()
    took = time.time() - start
    l.append(took)
    print(f"process took {took}")
        
def spatial_join(left, right, explain=False):
    res = left.alias("l").\
        join(right.alias("r"), f.expr("ST_Intersects(l.geometry, r.geometry)")).\
        select("l.id", "l.geometry")

    if explain:
        res.explain()

    res.write.mode("overwrite").format("noop").save()

    if explain:
        from sedona.utils.adapter import Adapter
        srdd = Adapter.toSpatialRdd(res, "geometry")
        print(srdd.getPartitioner())
    

In [6]:
with SedonaBenchmark({}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=10_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x==1))

    print(sum(container)/len(container))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/26 18:50:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/10/26 18:50:10 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/10/26 18:50:10 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/10/26 18:50:10 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/10/26 18:50:10 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.S2Geography.Geography, which is already registered.
25/10/26 18:50:10 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/10/26 18:50:10 WARN SimpleFunctionRegistry: The function st_envelop

process took 0.5338435173034668
== Physical Plan ==
*(3) Project [id#44L, geometry#45]
+- RangeJoin geometry#45: geometry, geometry#49: geometry, INTERSECTS
   :- *(1) Project [id#44L, geometry#45]
   :  +- *(1) Filter isnotnull(geometry#45)
   :     +- BatchScan spider[id#44L, geometry#45] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []
   +- *(2) Project [geometry#49]
      +- *(2) Filter isnotnull(geometry#49)
         +- BatchScan spider[id#48L, geometry#49] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []




/tmp/ipykernel_320/191528201.py:48: DeprecationWarning: Importing from 'sedona.utils' is deprecated. Please use 'sedona.spark.utils' instead.
  from sedona.utils.adapter import Adapter
/tmp/ipykernel_320/191528201.py:48: DeprecationWarning: Importing from 'sedona.utils.adapter' is deprecated. Please use 'sedona.spark.utils.adapter' instead.
  from sedona.utils.adapter import Adapter


SpatialPartitioner(name=None, jvm_partitioner=None)
process took 0.3887674808502197
process took 0.2623441219329834
process took 0.15990710258483887
process took 0.11682367324829102
process took 0.1042184829711914
process took 0.12341094017028809
process took 0.1218416690826416
process took 0.1086268424987793
process took 0.11799025535583496
0.20377740859985352
+-------+
|time[s]|
+-------+
|   0.53|
|   0.39|
|   0.26|
|   0.16|
|   0.12|
|    0.1|
|   0.12|
|   0.12|
|   0.11|
|   0.12|
+-------+



In [7]:
# removing optimizations

In [8]:
with SedonaBenchmark({"sedona.join.optimizationmode": "none"}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=10_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points))
    print(sum(container)/len(container))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

process took 16.781416416168213


process took 16.2198805809021


process took 16.40829038619995


process took 17.02281093597412


process took 17.089854955673218


process took 17.00855326652527


process took 17.36212682723999


process took 16.91432476043701


process took 16.499055862426758


process took 16.417631149291992
16.772394514083864
+-------+
|time[s]|
+-------+
|  16.78|
|  16.22|
|  16.41|
|  17.02|
|  17.09|
|  17.01|
|  17.36|
|  16.91|
|   16.5|
|  16.42|
+-------+



# broadcast vs non broadcast join

In [ ]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({"sedona.join.autoBroadcastJoinThreshold": "100MB", "spark.sql.autoBroadcastJoinThreshold": "50MB"}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=10_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=100_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        # uncomment it, if you want to have SQL version of it
        # polygons.createOrReplaceTempView("left")
        # points.createOrReplaceTempView("right")

        # sedona.sql(
        #     """
        #     SELECT
        #         /*+ BROADCASTJOIN (l) */
        #         l.id,
        #         l.geometry
        #     FROM right AS r
        #     JOIN left AS l ON ST_Intersects(r.geometry, l.geometry) 
        #     """
        # ).explain()

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, broadcast(points), explain=x == 0))
    print(sum(container)/len(container))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

In [ ]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({
    "sedona.join.autoBroadcastJoinThreshold": "-1",
    "spark.driver.memory": "12G",
    "spark.executor.memory": "16G",
}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=10_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=100_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    print(sum(container)/len(container))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

# Global index turn on and turn off for smaller datasets

In [11]:
from sedona.spark.core.SpatialRDD.spatial_rdd import SpatialRDD

In [12]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({"sedona.global.index": "False"}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=1000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=1000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    print(sum(container)/len(container))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

== Physical Plan ==
*(3) Project [id#753L, geometry#754]
+- RangeJoin geometry#754: geometry, geometry#758: geometry, INTERSECTS
   :- *(1) Project [id#753L, geometry#754]
   :  +- *(1) Filter isnotnull(geometry#754)
   :     +- BatchScan spider[id#753L, geometry#754] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []
   +- *(2) Project [geometry#758]
      +- *(2) Filter isnotnull(geometry#758)
         +- BatchScan spider[id#757L, geometry#758] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []


SpatialPartitioner(name=None, jvm_partitioner=None)
process took 0.15079379081726074
process took 0.0670161247253418
process took 0.06908559799194336
process took 0.06377840042114258
process took 0.05483889579772949
process took 0.06452417373657227
process took 0.05954694747924805
process took 0.059953927993774414
process took 0.07096052169799805
process took 0.056653738021850586
0.07171521186828614
+-------+
|

In [13]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=1000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=1000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    print(sum(container)/len(container))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

== Physical Plan ==
*(3) Project [id#979L, geometry#980]
+- RangeJoin geometry#980: geometry, geometry#984: geometry, INTERSECTS
   :- *(1) Project [id#979L, geometry#980]
   :  +- *(1) Filter isnotnull(geometry#980)
   :     +- BatchScan spider[id#979L, geometry#980] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []
   +- *(2) Project [geometry#984]
      +- *(2) Filter isnotnull(geometry#984)
         +- BatchScan spider[id#983L, geometry#984] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []


SpatialPartitioner(name=None, jvm_partitioner=None)
process took 0.11168789863586426
process took 0.06809616088867188
process took 0.06991887092590332
process took 0.08718347549438477
process took 0.05847740173339844
process took 0.04725527763366699
process took 0.06833791732788086
process took 0.18410325050354004
process took 0.10717248916625977
process took 0.05101180076599121
0.08532445430755616
+-------+
|t

# Global index turn on and turn off for larger datasets

In [ ]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({"sedona.global.index": "False"}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=100_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=1_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

In [15]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({"sedona.global.index": "True"}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=100_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=1_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    print(sum(container)/len(container))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

== Physical Plan ==
*(3) Project [id#1264L, geometry#1265]
+- RangeJoin geometry#1265: geometry, geometry#1269: geometry, INTERSECTS
   :- *(1) Project [id#1264L, geometry#1265]
   :  +- *(1) Filter isnotnull(geometry#1265)
   :     +- BatchScan spider[id#1264L, geometry#1265] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []
   +- *(2) Project [geometry#1269]
      +- *(2) Filter isnotnull(geometry#1269)
         +- BatchScan spider[id#1268L, geometry#1269] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []




SpatialPartitioner(name=None, jvm_partitioner=None)
process took 2.3350090980529785


process took 1.534728765487671


process took 1.278660774230957


process took 1.2597756385803223


...

process took 1.1852116584777832


process took 1.2878293991088867


process took 1.2084980010986328


process took 1.1110615730285645


process took 1.2478866577148438
1.3667003631591796
+-------+
|time[s]|
+-------+
|   2.34|
|   1.53|
|   1.28|
|   1.26|
|   1.22|
|   1.19|
|   1.29|
|   1.21|
|   1.11|
|   1.25|
+-------+



# Changing Number of partitions

In [16]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({"sedona.join.numpartition": 2, "app-name": "sedona"}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=100_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=1_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

== Physical Plan ==
*(3) Project [id#1490L, geometry#1491]
+- RangeJoin geometry#1491: geometry, geometry#1495: geometry, INTERSECTS
   :- *(1) Project [id#1490L, geometry#1491]
   :  +- *(1) Filter isnotnull(geometry#1491)
   :     +- BatchScan spider[id#1490L, geometry#1491] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []
   +- *(2) Project [geometry#1495]
      +- *(2) Filter isnotnull(geometry#1495)
         +- BatchScan spider[id#1494L, geometry#1495] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []




SpatialPartitioner(name=None, jvm_partitioner=None)
process took 3.2656984329223633


process took 2.280266284942627


process took 2.258615732192993


process took 2.238811492919922


...

process took 2.232933759689331


process took 2.2341461181640625


process took 2.268554210662842


process took 2.3197669982910156


process took 2.3856537342071533
+-------+
|time[s]|
+-------+
|   3.27|
|   2.28|
|   2.26|
|   2.24|
|   2.24|
|   2.23|
|   2.23|
|   2.27|
|   2.32|
|   2.39|
+-------+



In [ ]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({"sedona.join.numpartition": 200}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=1_000_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

In [ ]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({"sedona.join.numpartition": 5}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=1_000_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

In [ ]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({"sedona.join.numpartition": 100}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=1_000_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

In [ ]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({"sedona.join.numpartition": 60}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=1_000_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

In [ ]:
from pyspark.sql.functions import broadcast
import time
with SedonaBenchmark({"sedona.join.numpartition": 35, "spark.executor.instances": 3, "spark.executor.cores": 3}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=1_000_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

In [ ]:
import time
with SedonaBenchmark({"sedona.join.numpartition": 250}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=1_000_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

In [10]:
import time
with SedonaBenchmark({"sedona.join.numpartition": 1500}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=1_000_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

== Physical Plan ==
*(3) Project [id#946L, geometry#947]
+- RangeJoin geometry#947: geometry, geometry#951: geometry, INTERSECTS
   :- *(1) Project [id#946L, geometry#947]
   :  +- *(1) Filter isnotnull(geometry#947)
   :     +- BatchScan spider[id#946L, geometry#947] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []
   +- *(2) Project [geometry#951]
      +- *(2) Filter isnotnull(geometry#951)
         +- BatchScan spider[id#950L, geometry#951] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []




process took 42.06498169898987


process took 40.69003891944885


process took 39.95975613594055


process took 41.96866464614868


...

process took 39.82334852218628


process took 42.52166724205017


process took 41.12245488166809


process took 40.83388662338257


process took 40.29218149185181
+-------+
|time[s]|
+-------+
|  42.06|
|  40.69|
|  39.96|
|  41.97|
|   40.5|
|  39.82|
|  42.52|
|  41.12|
|  40.83|
|  40.29|
+-------+



In [ ]:
import time
with SedonaBenchmark({"sedona.join.numpartition": 10000}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=1_000_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

# changing the partitioning

In [ ]:
import time

with SedonaBenchmark({"sedona.join.gridtype": "quadtree"}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=100_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

In [26]:
import time
with SedonaBenchmark({"sedona.join.gridtype": "kdbtree"}) as sedona:
    container = []
    for x in range(10):
        point_params = GeometryGenerationParams(
            size=100_000,
            geometry_type="point"
        )
        
        polygon_params = GeometryGenerationParams(
            size=10_000_000,
            geometry_type="polygon"
        )
        
        polygons = create_geometries(polygon_params, point_params)
    
        points = create_geometries(point_params, polygon_params)

        check_time = time_it_with_container(container, lambda : spatial_join(polygons, points, x == 0))
    sedona.createDataFrame([{"time[s]": round(c, 2)} for c in container]).show()

== Physical Plan ==
*(3) Project [id#3332L, geometry#3333]
+- RangeJoin geometry#3333: geometry, geometry#3337: geometry, INTERSECTS
   :- *(1) Project [id#3332L, geometry#3333]
   :  +- *(1) Filter isnotnull(geometry#3333)
   :     +- BatchScan spider[id#3332L, geometry#3333] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []
   +- *(2) Project [geometry#3337]
      +- *(2) Filter isnotnull(geometry#3337)
         +- BatchScan spider[id#3336L, geometry#3337] class org.apache.sedona.sql.datasources.spider.SpiderScanBuilder$$anon$1 RuntimeFilters: []




process took 11.179224014282227


process took 11.652714729309082


process took 11.694740056991577


process took 12.64444613456726


...

process took 17.623416662216187


process took 15.683395385742188


process took 18.793715000152588


process took 20.21807050704956


process took 11.519941568374634
+-------+
|time[s]|
+-------+
|  11.18|
|  11.65|
|  11.69|
|  12.64|
|  12.01|
|  17.62|
|  15.68|
|  18.79|
|  20.22|
|  11.52|
+-------+

